In [1]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

# import logging
# logging.basicConfig(
#     level=logging.DEBUG,
#     format="%(asctime)s %(levelname)s %(name)s: %(message)s",
#     handlers=[
#         logging.FileHandler("curation.log"),
#         logging.StreamHandler(),  # keep console output too
#     ],
#     force=True,
# )

/Users/zakirov/Documents/GitHub/PerturbationCatalogue/.venv/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:157: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


# Download data


In [2]:
noncurated_path = "../non_curated/h5ad/gasperini_2019_atscale.h5ad"
download_file(
    url="https://exampledata.scverse.org/pertpy/gasperini_2019_atscale.h5ad",
    dest_path=noncurated_path
)

File ../non_curated/h5ad/gasperini_2019_atscale.h5ad already exists. Skipping download.


# Initialise the dataset object

In [3]:
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

Loading data from ../non_curated/h5ad/gasperini_2019_atscale.h5ad


# OBS slot curation

### Rename gene -> perturbation_name, barcode -> guide_sequence

In [4]:
cur_data.rename_columns(slot = 'obs',
                        name_dict = {"gene": "perturbation_name",
                                     "barcode": "guide_sequence"})
# add cell barcode
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.astype(str)
cur_data.adata.obs

Renamed columns in adata.obs: {'gene': 'perturbation_name', 'barcode': 'guide_sequence'}


,sample,total_umis,Size_Factor,perturbation_name,all_gene,guide_sequence,read_count,umi_count,proportion,guide_count,...,perturbation_type,celltype,organism,perturbation,nperts,ngenes,ncounts,percent_mito,percent_ribo,cell_barcode
AAACCTGAGAGGTACC,1A_1_SI-GA-E2,17572,1.009682,chr10.845_top_two_chr1.11183_top_two_chr1.1129...,chr10.845_top_two_chr1.11183_top_two_chr1.1129...,AGAAAGCTCCTCCAGTTCAC_TGATCGCTTTGACTGTGACA_ACAA...,14135.0,964.0,0.969819,67.0,...,CRISPR,lymphoblasts,human,chr10.845_top_two_chr1.11183_top_two_chr1.1129...,195,3549,17566.0,0.0,30.507799,AAACCTGAGAGGTACC
AAACCTGAGTCAATAG,1A_1_SI-GA-E2,8923,0.939677,chr1.12695_top_two_chr11.3294_top_two_chr1.679...,chr1.12695_top_two_chr11.3294_top_two_chr1.679...,GTAGAGCCTCCAGAACTGTG_AGGTTTATCCAGATGAACTG_CATC...,4329.0,293.0,0.844380,26.0,...,CRISPR,lymphoblasts,human,chr1.12695_top_two_chr11.3294_top_two_chr1.679...,68,2543,8917.0,0.0,28.955927,AAACCTGAGTCAATAG
AAACCTGCAAACAACA,1A_1_SI-GA-E2,14637,0.990803,ALDH1A2_TSS_BRI3_TSS_chr10.1918_top_two_chr10....,ALDH1A2_TSS_BRI3_TSS_chr10.1918_top_two_chr10....,CCAAGGCGTCCTCAGACCAG_AGCTCCAGGAAGGACCCCCG_TCAC...,12362.0,884.0,0.950538,61.0,...,CRISPR,lymphoblasts,human,ALDH1A2_BRI3_chr10.1918_top_two_chr10.350_top_...,167,3191,14626.0,0.0,26.548612,AAACCTGCAAACAACA
AAACCTGCACTTCTGC,1A_1_SI-GA-E2,22798,1.036578,C16orf91_TSS_chr1.11332_top_two_chr1.1933_top_...,C16orf91_TSS_chr1.11332_top_two_chr1.1933_top_...,GGCGTCAGTCGAGGAGTCAG_GCCAGCACTTCAGCTCACCG_GCTG...,7459.0,544.0,0.939551,39.0,...,CRISPR,lymphoblasts,human,C16orf91_chr1.11332_top_two_chr1.1933_top_two_...,109,4539,22783.0,0.0,20.616249,AAACCTGCACTTCTGC
AAACCTGCATGTAGTC,1A_1_SI-GA-E2,10136,0.952844,chr10.185_top_two_chr10.484_top_two_chr11.4167...,chr10.185_top_two_chr10.484_top_two_chr11.4167...,ATAAGGCACTCACATCCACC_GCTTGTCCCTAACACTCAGA_GGGC...,14831.0,1054.0,0.959927,37.0,...,CRISPR,lymphoblasts,human,chr10.185_top_two_chr10.484_top_two_chr11.4167...,107,2605,10124.0,0.0,32.418017,AAACCTGCATGTAGTC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCAGTACCTACA,2B_8_SI-GA-H9,17938,1.011811,BRIX1_TSS_chr10.2059_top_two_chr10.350_top_two...,BRIX1_TSS_chr10.2059_top_two_chr10.350_top_two...,AGGAGGCCAAGAGCGCGGGG_TGATTGAAGGAGGCTCCCCA_GGTA...,4570.0,394.0,0.856522,31.0,...,CRISPR,lymphoblasts,human,BRIX1_chr10.2059_top_two_chr10.350_top_two_chr...,85,3451,17934.0,0.0,31.733021,TTTGTCAGTACCTACA
TTTGTCAGTATCACCA-1,2B_8_SI-GA-H9,16543,1.003448,chr10.221_top_two_chr11.3853_top_two_chr11.498...,chr10.221_top_two_chr11.3853_top_two_chr11.498...,CCTGCAACTGTCTATGGCCT_TCAGTGGGTGAGTCTTCAGG_TGAG...,4311.0,387.0,0.861915,33.0,...,CRISPR,lymphoblasts,human,chr10.221_top_two_chr11.3853_top_two_chr11.498...,92,3693,16542.0,0.0,24.694716,TTTGTCAGTATCACCA-1
TTTGTCAGTTCAGACT-1,2B_8_SI-GA-H9,15009,0.993396,chr12.3400_top_two_chr4.1281_second_two_ID3_TSS,chr12.3400_top_two_chr4.1281_second_two_ID3_TSS,CCAGTTGCTAGGCAGGACAG_CTTTCAGGCAGGAATCTGAG_TGGT...,669.0,57.0,0.876923,3.0,...,CRISPR,lymphoblasts,human,chr12.3400_top_two_chr4.1281_second_two_ID3,7,3162,14992.0,0.0,32.123799,TTTGTCAGTTCAGACT-1
TTTGTCAGTTCTGTTT,2B_8_SI-GA-H9,5149,0.882878,chr7.1014_top_two,chr7.1014_top_two,CACCCCCCAGAGATAAGAGA,77.0,7.0,0.086420,1.0,...,CRISPR,lymphoblasts,human,chr7.1014_top_two,3,1392,5149.0,0.0,35.074772,TTTGTCAGTTCTGTTT


### Show unique perturbations

In [5]:
cur_data.show_unique(slot = 'obs', column = 'perturbation_name')

Unique values in adata.obs.perturbation_name: 203515
--------------------------------------------------
{

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



### Add guide RNA information

In [ ]:
import re

# define regex pattern for splitting
controls_regex_pattern = r"(top_two|second_two|TSS|bassik_mch|pos_control_HBE1_tss_Klann_mosaic|pos_control_HBE1_tss_Klann_mosaic|pos_control_HS2_Klann_mosaic|pos_control_Klannchr1_HBG1_HBG1_tss_both|pos_control_Klannchr1_HS3|pos_control_Klannchr1_HS4|pos_control_Klannchr_HS1|pos_control_KlannHS2g_HS2_A|pos_control_KlannHS2g_HS2_B|pos_control_mosaic_HB_HBE1_tss_A|pos_control_mosaic_HB_HBE1_tss_B|scrambled_\d+|random_\d+)"
# split the gene names based on the regex pattern
gene_split = [re.sub(controls_regex_pattern, r"\1\n", e).split('\n') for e in cur_data.adata.obs['perturbation_name']]
# clean up the split parts
gene_split = [[i.strip('_').replace('_TSS', '') for i in e if i != ''] for e in gene_split]
# join the split parts with '|'
gene_split = ['|'.join(e) for e in gene_split]

cur_data.adata.obs['gene_split'] = gene_split

cur_data.adata.obs['gene_split']

AAACCTGAGAGGTACC      chr10.845_top_two|chr1.11183_top_two|chr1.1129...
AAACCTGAGTCAATAG      chr1.12695_top_two|chr11.3294_top_two|chr1.679...
AAACCTGCAAACAACA      ALDH1A2|BRI3|chr10.1918_top_two|chr10.350_top_...
AAACCTGCACTTCTGC      C16orf91|chr1.11332_top_two|chr1.1933_top_two|...
AAACCTGCATGTAGTC      chr10.185_top_two|chr10.484_top_two|chr11.4167...
                                            ...                        
TTTGTCAGTACCTACA      BRIX1|chr10.2059_top_two|chr10.350_top_two|chr...
TTTGTCAGTATCACCA-1    chr10.221_top_two|chr11.3853_top_two|chr11.498...
TTTGTCAGTTCAGACT-1          chr12.3400_top_two|chr4.1281_second_two|ID3
TTTGTCAGTTCTGTTT                                      chr7.1014_top_two
TTTGTCATCAAAGTAG-1    chr11.2790_top_two|chr12.306_top_two|chr12.474...
Name: gene_split, Length: 207324, dtype: object

In [ ]:
# download supplementary files on at-scale guide annotations
download_file(
    url="https://ars.els-cdn.com/content/image/1-s2.0-S009286741831554X-mmc2.xlsx",
    dest_path="../supplementary/gasperini_2019_atscale_supp.xlsx"
)
download_file(
    url="https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE120861&format=file&file=GSE120861%5Fgrna%5Fgroups%2Eat%5Fscale%2Etxt%2Egz",
    dest_path="../supplementary/gasperini_2019_atscale_supp.txt.gz"
)
# load guide info
guide_info_df = pd.read_excel("../supplementary/gasperini_2019_atscale_supp.xlsx", sheet_name="S2A_AtScale_library_gRNA.cs", skiprows=0)
guide_info_df2 = pd.read_table("../supplementary/gasperini_2019_atscale_supp.txt.gz", sep='\t', header=None, names=['target', 'guide_sequence'])
# add coordinate info
guide_info_df['start.candidate_enhancer'] = guide_info_df['start.candidate_enhancer'].astype(str).str.replace('.0', '')
guide_info_df['stop.candidate_enhancer'] = guide_info_df['stop.candidate_enhancer'].astype(str).str.replace('.0', '')
guide_info_df['coord'] = (guide_info_df['chr.candidate_enhancer']+ ':' +
                              guide_info_df['start.candidate_enhancer'] + '-' +
                              guide_info_df['stop.candidate_enhancer'])
guide_info_df.loc[guide_info_df['coord'] == 'nan:nan-nan', 'coord'] = ''
guide_info_df = guide_info_df[['Spacer', 'coord']]#.fillna('')#.dropna()
# remove _TSS from target names
guide_info_df2['target'] = guide_info_df2['target'].str.replace('_TSS', '')
guide_info_df2 = guide_info_df2.merge(guide_info_df, left_on='guide_sequence', right_on='Spacer', how='left').drop(columns=['Spacer'])

guide_info_df2


File ../supplementary/gasperini_2019_atscale_supp.xlsx already exists. Skipping download.
File ../supplementary/gasperini_2019_atscale_supp.txt.gz already exists. Skipping download.


,target,guide_sequence,coord
0,SH3BGRL3,AAACCGCTCCCGAGCACGGG,NaN
1,MTRNR2L8,AAATAGTGGGAAGATTCGTG,NaN
2,FAM83A,AACACACCACGGAGGAGTGG,NaN
3,ZNF593,AACAGCCCGGCCGGCCAAGG,NaN
4,ATPIF1,AACGAGAGACTGCTTGCTGG,NaN
...,...,...,...
13184,random_5,TTAATTCCTCTGGCGCCGCT,NaN
13185,random_14,TTAGCTTTGAAAACACACAC,NaN
13186,random_2,TTATCTCTATTTGACAGACG,NaN
13187,scrambled_23,TTGGGAATGCGAGGCAAAGG,NaN


In [8]:
# get gene ont
gene_ont = cur_data.gene_ont.dropna(subset='synonym')
# remove unnecessary mappings based on chr
gene_ont = gene_ont[gene_ont['chromosome_name'].isin(
    [str(i) for i in range(1, 23)] + ["X", "Y", "MT"]
)]

# create a mapping dict name:coord
enh_mapping_dict = (
    guide_info_df2[guide_info_df2['target'].str.startswith('chr')][['target', 'coord']]
    .drop_duplicates()
    .set_index('target')['coord']
    .to_dict()
)

# init enhancer mapping df
enh_mapping_df = pd.DataFrame(columns=gene_ont.columns)
enh_mapping_df['synonym'] = enh_mapping_dict.keys()
enh_mapping_df['ensembl_gene_id'] = enh_mapping_dict.keys()
enh_mapping_df['ensembl_gene_id'] = enh_mapping_df['ensembl_gene_id'].str.replace('chr', 'enh_chr_')
enh_mapping_df['gene_symbol'] = enh_mapping_df['ensembl_gene_id']
enh_mapping_df['gene_coord'] = enh_mapping_dict.values()
enh_mapping_df['chromosome_name'] = enh_mapping_df['gene_coord'].str.split(':').str[0].str.replace('chr', '')
enh_mapping_df['biotype'] = 'enhancer'
enh_mapping_df['synonym_type'] = 'symbol_syn'
enh_mapping_df['description'] = 'enhancer'

# concat to gene_ont df
gene_ont = pd.concat([gene_ont, enh_mapping_df], axis=0, ignore_index=True)

# replace cur_data.gene_ont with the updated one
cur_data.gene_ont = gene_ont

gene_ont.tail()

,ensembl_gene_id,gene_symbol,chromosome_name,gene_coord,biotype,description,synonym_type,synonym
520416,enh_chr_X.938_top_two,enh_chr_X.938_top_two,X,chrX:48489654-48490612,enhancer,enhancer,symbol_syn,chrX.938_top_two
520417,enh_chr_X.94_top_two,enh_chr_X.94_top_two,X,chrX:2953853-2954104,enhancer,enhancer,symbol_syn,chrX.94_top_two
520418,enh_chr_X.952_top_two,enh_chr_X.952_top_two,X,chrX:48641145-48641725,enhancer,enhancer,symbol_syn,chrX.952_top_two
520419,enh_chr_X.952_second_two,enh_chr_X.952_second_two,X,chrX:48641145-48641725,enhancer,enhancer,symbol_syn,chrX.952_second_two
520420,enh_chr_X.963_top_two,enh_chr_X.963_top_two,X,chrX:48726316-48726706,enhancer,enhancer,symbol_syn,chrX.963_top_two


In [9]:
# replace controls in gene_split with mappable ones
cur_data.adata.obs['gene_split'] = (
    cur_data.adata.obs['gene_split']
        .replace(
        {r'random_\d+': 'control_genedesert',
         r'scrambled_\d+': 'control_nontargeting',
         r'pos_control_\w+': 'control_positive'
         },
        regex=True
    )
)


In [12]:
cur_data.adata.obs

,Size_Factor,cancer,id,nperts,umi_count,within_batch_chip,perturbation_name,percent_mito,prep_batch,all_gene,...,percent.mito,sample_directory,cell_barcode,perturbation,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index
index,,,,,,,,,,,,,,,,,,,,,
0,1.009682,True,1A_1,195,964.0,within_batch_chip_A,chr10.845_top_two_chr1.11183_top_two_chr1.1129...,0.0,prep_batch_1,chr10.845_top_two_chr1.11183_top_two_chr1.1129...,...,0.058787,1A_1_SI-GA-E2,AAACCTGAGAGGTACC,chr10.845_top_two_chr1.11183_top_two_chr1.1129...,enh_chr_10.845_top_two|enh_chr_1.11183_top_two...,ENH_CHR_10.845_TOP_TWO|ENH_CHR_1.11183_TOP_TWO...,enhancer|enhancer|enhancer|enhancer|enhancer|e...,chr10:23344953-23345044|chr1:205720419-2057208...,10|1|1|11|1|1|11|12|12|12|13|14|14|1|15|15|15|...,AAACCTGAGAGGTACC|AAACCTGAGAGGTACC|AAACCTGAGAGG...
1,0.939677,True,1A_1,68,293.0,within_batch_chip_A,chr1.12695_top_two_chr11.3294_top_two_chr1.679...,0.0,prep_batch_1,chr1.12695_top_two_chr11.3294_top_two_chr1.679...,...,0.036087,1A_1_SI-GA-E2,AAACCTGAGTCAATAG,chr1.12695_top_two_chr11.3294_top_two_chr1.679...,enh_chr_1.12695_top_two|enh_chr_11.3294_top_tw...,ENH_CHR_1.12695_TOP_TWO|ENH_CHR_11.3294_TOP_TW...,enhancer|enhancer|enhancer|enhancer|enhancer|e...,chr1:237077735-237078115|chr11:65459793-654603...,1|11|1|17|20|20|2|2|2|2|3|3|5|6|6|6|6|6|6|7|7|...,AAACCTGAGTCAATAG|AAACCTGAGTCAATAG|AAACCTGAGTCA...
2,0.990803,True,1A_1,167,884.0,within_batch_chip_A,ALDH1A2_TSS_BRI3_TSS_chr10.1918_top_two_chr10....,0.0,prep_batch_1,ALDH1A2_TSS_BRI3_TSS_chr10.1918_top_two_chr10....,...,0.069823,1A_1_SI-GA-E2,AAACCTGCAAACAACA,ALDH1A2_BRI3_chr10.1918_top_two_chr10.350_top_...,ENSG00000128918|ENSG00000164713|enh_chr_10.191...,ALDH1A2|BRI3|ENH_CHR_10.1918_TOP_TWO|ENH_CHR_1...,protein_coding|protein_coding|enhancer|enhance...,chr15:57953424-58497866;-1|chr7:98252379-98310...,15|7|10|10|10|10|1|11|11|11|11|12|1|15|16|16|1...,AAACCTGCAAACAACA|AAACCTGCAAACAACA|AAACCTGCAAAC...
3,1.036578,True,1A_1,109,544.0,within_batch_chip_A,C16orf91_TSS_chr1.11332_top_two_chr1.1933_top_...,0.0,prep_batch_1,C16orf91_TSS_chr1.11332_top_two_chr1.1933_top_...,...,0.026187,1A_1_SI-GA-E2,AAACCTGCACTTCTGC,C16orf91_chr1.11332_top_two_chr1.1933_top_two_...,ENSG00000174109|enh_chr_1.11332_top_two|enh_ch...,UQCC4|ENH_CHR_1.11332_TOP_TWO|ENH_CHR_1.1933_T...,protein_coding|enhancer|enhancer|enhancer|enha...,chr16:1419752-1420756;-1|chr1:207378404-207378...,16|1|1|14|15|15|16|16|16|1|1|18|1|20|20|22|3|3...,AAACCTGCACTTCTGC|AAACCTGCACTTCTGC|AAACCTGCACTT...
4,0.952844,True,1A_1,107,1054.0,within_batch_chip_A,chr10.185_top_two_chr10.484_top_two_chr11.4167...,0.0,prep_batch_1,chr10.185_top_two_chr10.484_top_two_chr11.4167...,...,0.007991,1A_1_SI-GA-E2,AAACCTGCATGTAGTC,chr10.185_top_two_chr10.484_top_two_chr11.4167...,enh_chr_10.185_top_two|enh_chr_10.484_top_two|...,ENH_CHR_10.185_TOP_TWO|ENH_CHR_10.484_TOP_TWO|...,enhancer|enhancer|enhancer|enhancer|enhancer|e...,chr10:4983650-4984221|chr10:12105451-12105804|...,10|10|11|12|13|14|15|15|1|18|18|18|1|18|20|20|...,AAACCTGCATGTAGTC|AAACCTGCATGTAGTC|AAACCTGCATGT...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207319,1.011811,True,2B_8,85,394.0,within_batch_chip_B,BRIX1_TSS_chr10.2059_top_two_chr10.350_top_two...,0.0,prep_batch_2,BRIX1_TSS_chr10.2059_top_two_chr10.350_top_two...,...,0.060598,2B_8_SI-GA-H9,TTTGTCAGTACCTACA,BRIX1_chr10.2059_top_two_chr10.350_top_two_chr...,ENSG00000113460|enh_chr_10.2059_top_two|enh_ch...,BRIX1|ENH_CHR_10.2059_TOP_TWO|ENH_CHR_10.350_T...,protein_coding|enhancer|enhancer|enhancer|enha...,chr5:34915273-34926622;1|chr10:70821228-708218...,5|10|10|1|1|1|11|12|14|14|1|15|16|19|19|20|20|...,TTTGTCAGTACCTACA|TTTGTCAGTACCTACA|TTTGTCAGTACC...
207320,1.003448,True,2B_8,92,387.0,within_batch_chip_B,chr10.221_top_two_chr11.3853_top_two_chr11.498...,0.0,prep_batch_2,chr10.221_top_two_chr11.3853_top_two_chr11.498...,...,0.057789,2B_8_SI-GA-H9,TTTG

### Standardise perturbation targets

In [11]:
cur_data.standardize_genes(
    slot='obs',
    input_column='gene_split',
    input_column_type='gene_symbol',
    multiple_entries=True,
    multiple_entries_sep='|'
)

Mapping gene symbols: 100%|██████████████████████████████████| 6542/6542 [00:00<00:00, 46474.56it/s]


--------------------------------------------------
Successfully mapped 6540 out of 6542 gene symbols.
--------------------------------------------------
Couldn't map gene symbols: ['bassik_mch', 'MTRNR2L8']
--------------------------------------------------
Collapsed column positional_index using separator |


/Users/zakirov/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [14]:
# clean the mapped names ensg and symbol columns
controls_regex_pattern = r"(_top_two|_second_two)"

cur_data.adata.obs['perturbed_target_ensg'] = cur_data.adata.obs['perturbed_target_ensg'].str.replace(controls_regex_pattern, '', regex=True)
cur_data.adata.obs['perturbed_target_symbol'] = cur_data.adata.obs['perturbed_target_symbol'].str.replace(controls_regex_pattern.upper(), '', regex=True)


### Add `perturbed_target_number` column

In [17]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

Counted entries in column perturbed_target_symbol of adata.obs and stored in perturbed_target_number


### Encode chromosomes as integers

In [18]:
cur_data.chromosome_encoding()

Chromosome encoding applied to perturbed_target_chromosome in adata.obs and stored as 'perturbed_target_chromosome_encoding'.


### Add replicate information

In [23]:
cur_data.adata.obs = cur_data.adata.obs.rename(
    columns={
        'prep_batch': 'technical_replicate'
    }
)

### Add guide sequence information

In [34]:
cur_data.adata.obs['guide_sequence'] = cur_data.adata.obs['guide_sequence'].str.replace('_', '|')

### Add metadata

In [25]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        #----- dataset -----#
        "dataset_id": cur_data.dataset_id,
        #----- sample -----#
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        #----- perturbation type -----#
        "perturbation_type_label": "CRISPRi",
        "perturbation_type_id": None,
        #----- data modality -----#
        "data_modality": "Perturb-seq", # different from "method_name_label"; more general term - choice of CRISPR, MAVE and Perturb-seq
        #----- significance -----#
        "significant": None,
        "significance_criteria": None,
        #----- score interpretation -----#
        "score_interpretation": None,
        #----- treatment -----#
        "treatment_label": None,
        "treatment_id": None,
        #----- replicate -----#
        # "technical_replicate": None,
        "biological_replicate": None,
        #----- model system -----#
        "model_system_label": "cell_line",
        "model_system_id": None,
        #----- tissue -----#
        "tissue": "blood",
        #----- cell line -----#
        "cell_line_label": "K 562 cell",
        # "cell_line_id": None,
        #----- cell type -----#
        "cell_type_label": "lymphoblast",
        # "cell_type_id": ,
        #----- disease -----#
        "disease_label": "chronic myelogenous leukemia, BCR-ABL1 positive",
        "disease_id": "MONDO:0011996",
        #----- timepoint -----#
        "timepoint": "P10DT0H0M0S",
        #----- species -----#
        "species": "Homo sapiens",
        #----- sex -----#
        "sex_label": "female",
        "sex_id": None,
        #----- developmental stage -----#
        "developmental_stage_label": "adult",
        "developmental_stage_id": None,
        #----- study metadata -----#
        "study_title": "A Genome-wide Framework for Mapping Gene Regulation via Cellular Genetic Screens",
        "study_uri": "https://doi.org/10.1016/j.cell.2018.11.029",
        "study_year": 2019,
        #----- authors -----#
        "first_author": "Molly Gasperini",
        "last_author": "Jay Shendure",
        #----- experiment metadata -----#
        "experiment_title": "Perturb-seq CRISPRi screen in K562 cells to explore the targets of over 5,779 candidate enhancers.",
        "experiment_summary": """
            Several methods were used to select a set of 5,799 candidate enhancers. GuideRNAs targeting these candidate enhancers, as well a selection of non-targeting , gene-desert and positive controls (targeting globin TSS and enhancers + 381 gene TSSs) were cloned into a CRISPRi-optimised CROP-seq vector. After producing the lentiviral library, K562 cells stably expressing the dCas9-BFP-KRAB were transduced with the library at A very high MOI=~28 (median 28 ± 15.3 gRNAs identified per cell) and cultured for 10 days, at which point the cells were harvested for sequencing using Illumina NovaSeq 6000.
        """,
        #----- number of perturbed targets/samples -----#
        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],
        #----- library generation type -----#
        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",
        #----- library generation method -----#
        "library_generation_method_id": "EFO:0022895",
        "library_generation_method_label": "dCas9-KRAB",
        #----- enzyme and library delivery method -----#
        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",
        #----- enzyme and library integration state -----#
        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",
        #----- enzyme and library expression control -----#
        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",
        #----- library name and URI and manufacturer -----#
        "library_name": "custom",
        "library_uri": None,
        "library_manufacturer": "Shendure lab",
        #----- library format -----#
        "library_format_id": None,
        "library_format_label": "pooled",
        #----- library scope -----#
        "library_scope_id": None,
        "library_scope_label": "focused",
        #----- library perturbation type -----#
        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "inhibition",
        #----- library additional metadata -----#
        "library_lentiviral_generation": "3",
        "library_grnas_per_target": "2",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()), # for CRISPR/Perturb-seq
        "library_total_variants": None, # for MAVE
        #----- readout dimensionality -----#
        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",
        #---- readout type -----#
        "readout_type_id": None,
        "readout_type_label": "transcriptomic",
        #----- readout technology -----#
        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",
        #----- method -----#
        "method_name_id": None,
        "method_name_label": "Perturb-seq", # different from "data_modality"; more specific term - specific name of the technique
        "method_uri": None,
        #----- sequencing library kit -----#
        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "10x Genomics Single Cell 3-prime v2",
        #----- sequencing platform -----#
        "sequencing_platform_id": None,
        "sequencing_platform_label": "Illumina NovaSeq 6000",
        #----- sequencing strategy -----#
        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",
        #----- software used for counts-----#
        "software_counts_id": None,
        "software_counts_label": "CellRanger",
        #----- software used for analysis -----#
        "software_analysis_id": None,
        "software_analysis_label": "Seurat",
        #----- reference genome -----#
        "reference_genome_id": None,
        "reference_genome_label": "GRCh37",
        #----- license -----#
        "license_label": "free to use license",
        "license_id": "SWO:1000061",
        #----- external datasets -----#
        "associated_datasets": json.dumps([
            {
                "dataset_accession": "gasperini_2019_atscale.h5ad",
                "dataset_uri": "https://exampledata.scverse.org/pertpy/gasperini_2019_atscale.h5ad",
                "dataset_description": "Raw counts - .h5ad file from pertpy",
                "dataset_file_name": "gasperini_2019_atscale.h5ad",
            },
            {
                "dataset_accession": "GSE120861",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE120861",
                "dataset_description": "Raw counts - GEO entry",
                "dataset_file_name": "GSE120861_at_scale_screen.*",
            }
        ])
    }
)

Column dataset_id added to adata.obs
Column sample_id added to adata.obs
Column perturbation_type_label added to adata.obs
Column perturbation_type_id added to adata.obs
Column data_modality added to adata.obs
Column significant added to adata.obs
Column significance_criteria added to adata.obs
Column score_interpretation added to adata.obs
Column treatment_label added to adata.obs
Column treatment_id added to adata.obs
Column biological_replicate added to adata.obs
Column model_system_label added to adata.obs
Column model_system_id added to adata.obs
Column tissue added to adata.obs
Column cell_line_label added to adata.obs
Column cell_type_label added to adata.obs
Column disease_label added to adata.obs
Column disease_id added to adata.obs
Column timepoint added to adata.obs
Column species added to adata.obs
Column sex_label added to adata.obs
Column sex_id added to adata.obs
Column developmental_stage_label added to adata.obs
Column developmental_stage_id added to adata.obs
Column s

### Curate tissue information


In [26]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

Mapped 1 tissue ontology terms from `tissue` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
  input_column input_column_lower name_lower     ontology_id
0        blood              blood      blood  UBERON:0000178
--------------------------------------------------


/Users/zakirov/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell type information


In [27]:
cur_data.standardize_ontology(
    input_column='cell_type_label',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

Mapped 1 cell_type ontology terms from `cell_type_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
  input_column input_column_lower   name_lower ontology_id
0  lymphoblast        lymphoblast  lymphoblast  CL:0017005
--------------------------------------------------
Overwriting column cell_type_label in adata.obs


/Users/zakirov/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell line information

In [28]:
cur_data.standardize_ontology(
    input_column='cell_line_label',
    column_type='term_name',
    ontology_type='cell_line',
    overwrite=True
)

Mapped 1 cell_line ontology terms from `cell_line_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
  input_column input_column_lower  name_lower  ontology_id
0   K 562 cell         k 562 cell  k 562 cell  CLO:0007050
--------------------------------------------------
Overwriting column cell_line_label in adata.obs


/Users/zakirov/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate disease information

In [29]:
cur_data.standardize_ontology(
    input_column='disease_label',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

Mapped 1 disease ontology terms from `disease_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
                                      input_column  \
0  chronic myelogenous leukemia, BCR-ABL1 positive   

                                input_column_lower  \
0  chronic myelogenous leukemia, bcr-abl1 positive   

                                        name_lower    ontology_id  
0  chronic myelogenous leukemia, bcr-abl1 positive  MONDO:0011996  
--------------------------------------------------
Overwriting column disease_label in adata.obs
Overwriting column disease_id in adata.obs


/Users/zakirov/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Match schema column order

In [30]:
cur_data.match_schema_columns(slot='obs')

Matched columns of adata.obs to the obs_schema.


### Validate obs metadata

In [31]:
cur_data.validate_data(slot='obs', verbose=True)

,dataset_id,sample_id,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,perturbed_target_number,...,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
0,gasperini_2019_atscale,1,Perturb-seq,<NA>,<NA>,chr10.845_top_two_chr1.11183_top_two_chr1.1129...,chr10:23344953-23345044|chr1:205720419-2057208...,10|1|1|11|1|1|11|12|12|12|13|14|14|1|15|15|15|...,0,67,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh37,"[{""dataset_accession"": ""gasperini_2019_atscale...",free to use license,SWO:1000061
1,gasperini_2019_atscale,2,Perturb-seq,<NA>,<NA>,chr1.12695_top_two_chr11.3294_top_two_chr1.679...,chr1:237077735-237078115|chr11:65459793-654603...,1|11|1|17|20|20|2|2|2|2|3|3|5|6|6|6|6|6|6|7|7|...,0,25,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh37,"[{""dataset_accession"": ""gasperini_2019_atscale...",free to use license,SWO:1000061
2,gasperini_2019_atscale,3,Perturb-seq,<NA>,<NA>,ALDH1A2_TSS_BRI3_TSS_chr10.1918_top_two_chr10....,chr15:57953424-58497866;-1|chr7:98252379-98310...,15|7|10|10|10|10|1|11|11|11|11|12|1|15|16|16|1...,0,61,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh37,"[{""dataset_accession"": ""gasperini_2019_atscale...",free to use license,SWO:1000061
3,gasperini_2019_atscale,4,Perturb-seq,<NA>,<NA>,C16orf91_TSS_chr1.11332_top_two_chr1.1933_top_...,chr16:1419752-1420756;-1|chr1:207378404-207378...,16|1|1|14|15|15|16|16|16|1|1|18|1|20|20|22|3|3...,0,39,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh37,"[{""dataset_accession"": ""gasperini_2019_atscale...",free to use license,SWO:1000061
4,gasperini_2019_atscale,5,Perturb-seq,<NA>,<NA>,chr10.185_top_two_chr10.484_top_two_chr11.4167...,chr10:4983650-4984221|chr10:12105451-12105804|...,10|10|11|12|13|14|15|15|1|18|18|18|1|18|20|20|...,0,37,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh37,"[{""dataset_accession"": ""gasperini_2019_atscale...",free to use license,SWO:1000061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207319,gasperini_2019_atscale,207320,Perturb-seq,<NA>,<NA>,BRIX1_TSS_chr10.2059_top_two_chr10.350_top_two...,chr5:34915273-34926622;1|chr10:70821228-708218...,5|10|10|1|1|1|11|12|14|14|1|15|16|19|19|20|20|...,0,31,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh37,"[{""dataset_accession"": ""gasperini_2019_atscale...",free to use license,SWO:1000061
207320,gasperini_2019_atscale,207321,Perturb-seq,<NA>,<NA>,chr10.221_top_two_chr11.3853_top_two_chr11.498...,chr10:5520856-5520965|chr11:69796098-69797225|...,10|11|11|12|12|12|13|14|17|19|19|19|1|20|20|2|...,0,33,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh37,"[{""dataset_accession"": ""gasperini_2019_atscale...",free to use license,SWO:1000061
207321,gasperini_2019_atscale,207322,Perturb-seq,<NA>,<NA>,chr12.3400_top_two_chr4.1281_second_two_ID3_TSS,chr12:108512970-108513426|chr4:38244351-382452...,12|4|1,0,3,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh37,"[{""dataset_accession"": ""gasperini_2019_atscale...",free to use license,SWO:1000061
207322,gasperini_2019_atscale,207323,Perturb-seq,<NA>,<NA>,chr7.1014_top_two,chr7:13888984-13889047,7,7,1,...,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh37,"[{""dataset_accession"": ""gasperini_2019_atscale...",free to use license,SWO:1000061


# VAR slot curation

### Standardise genes

In [39]:
cur_data.adata.var['gene_name'] = cur_data.adata.var.index
cur_data.adata.var

,ensembl_id,ncounts,ncells,gene_name
gene_symbol,,,,
ENSG00000238009,ENSG00000238009,4062.0,3970,ENSG00000238009
ENSG00000237683,ENSG00000237683,13643.0,12625,ENSG00000237683
ENSG00000228463,ENSG00000228463,135915.0,90115,ENSG00000228463
ENSG00000237094,ENSG00000237094,4796.0,4706,ENSG00000237094
ENSG00000235373,ENSG00000235373,10491.0,10072,ENSG00000235373
...,...,...,...,...
ENSG00000215689,ENSG00000215689,6117.0,5943,ENSG00000215689
ENSG00000215781,ENSG00000215781,43665.0,36415,ENSG00000215781
ENSG00000220023,ENSG00000220023,201108.0,113584,ENSG00000220023


In [40]:
cur_data.standardize_genes(
    slot="var",
    input_column="ensembl_id",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

Missing Ensembl IDs: ['ENSG00000271976', 'ENSG00000223797', 'ENSG00000242861', 'ENSG00000272520', 'ENSG00000273314', 'ENSG00000206176', 'ENSG00000267283', 'ENSG00000232063', 'ENSG00000254690', 'ENSG00000269973', 'ENSG00000273295', 'ENSG00000273055', 'ENSG00000269936', 'ENSG00000215615', 'ENSG00000148362', 'ENSG00000162290', 'ENSG00000227540', 'ENSG00000267199', 'ENSG00000260267', 'ENSG00000261542', 'ENSG00000226377', 'ENSG00000206129', 'ENSG00000272969', 'ENSG00000272416', 'ENSG00000215066', 'ENSG00000225643', 'ENSG00000260500', 'ENSG00000272593', 'ENSG00000259959', 'ENSG00000269068', 'ENSG00000261158', 'ENSG00000251867', 'ENSG00000184886', 'ENSG00000272960', 'ENSG00000233903', 'ENSG00000236030', 'ENSG00000228779', 'ENSG00000237481', 'ENSG00000259799', 'ENSG00000271912', 'ENSG00000262967', 'ENSG00000257207', 'ENSG00000238261', 'ENSG00000272053', 'ENSG00000272827', 'ENSG00000262758', 'ENSG00000272579', 'ENSG00000271204', 'ENSG00000273271', 'ENSG00000248835', 'ENSG00000271308', 'ENSG0000

/Users/zakirov/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [41]:
cur_data.adata.var

,gene_name,ensembl_id,ncells,ncounts,ensembl_gene_id,gene_symbol,original_index
index,,,,,,,
0,ENSG00000238009,ENSG00000238009,3970,4062.0,ENSG00000241860,NaN,ENSG00000238009
1,ENSG00000237683,ENSG00000237683,12625,13643.0,ENSG00000237683,NaN,ENSG00000237683
2,ENSG00000228463,ENSG00000228463,90115,135915.0,ENSG00000228463,NaN,ENSG00000228463
3,ENSG00000237094,ENSG00000237094,4706,4796.0,ENSG00000237094,NaN,ENSG00000237094
4,ENSG00000235373,ENSG00000235373,10072,10491.0,ENSG00000235373,NaN,ENSG00000235373
...,...,...,...,...,...,...,...
13130,ENSG00000215689,ENSG00000215689,5943,6117.0,ENSG00000215689,NaN,ENSG00000215689
13131,ENSG00000215781,ENSG00000215781,36415,43665.0,ENSG00000215781,NaN,ENSG00000215781
13132,ENSG00000220023,ENSG00000220023,113584,201108.0,ENSG00000220023,NaN,ENSG00000220023


### Validate var metadata

In [42]:
cur_data.validate_data(slot='var')

,ensembl_gene_id,gene_symbol
index,,
0,ENSG00000241860,NaN
1,ENSG00000237683,NaN
2,ENSG00000228463,NaN
3,ENSG00000237094,NaN
4,ENSG00000235373,NaN
...,...,...
13130,ENSG00000215689,NaN
13131,ENSG00000215781,NaN
13132,ENSG00000220023,NaN


# Save the dataset

In [43]:
cur_data.save_curated_data_h5ad()

/Users/zakirov/Documents/GitHub/PerturbationCatalogue/data_exploration/curation_tools/curation_tools.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adata.obs = adata.obs.fillna(value=np.nan)


✅ Curated h5ad data saved to ../curated/h5ad/gasperini_2019_atscale_curated.h5ad


In [44]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

✅ Metadata saved to ../curated/parquet/gasperini_2019_atscale_curated_metadata.parquet


# Upload to BigQuery

In [46]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/gasperini_2019_atscale_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

Staging table: loading `.parquet` file ../curated/parquet/gasperini_2019_atscale_curated_metadata.parquet to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging...
Staging table: loaded 207324 rows to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Staging table: added ingested_at timestamp column to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Merge completed: staging → prj-ext-dev-pertcat-437314.perturb_seq.metadata with type-safe casting.
Staging table: deleted prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging


# Upload to GC Storage

In [45]:
!gcloud storage cp ../curated/h5ad/gasperini_2019_atscale_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://../curated/h5ad/gasperini_2019_atscale_curated.h5ad to gs://perturbation-catalogue-lake/perturbseq/curated/gasperini_2019_atscale_curated.h5ad
  Completed files 32/1 | 5.9GiB/5.9GiB | 11.5MiB/s                             

Average throughput: 19.5MiB/s


Updates are available for some Google Cloud CLI components.  To install them,